# 完整流程示例（对照 CLI）

路径用 `meteva_base.get_path`。数据根目录为 `resource/data/`。

风速用最新的 0.1° 等高层结果：`wind/YYYY/YYYYMMDD/YYYYMMDDHH.TTT.nc`。订正前把风速插到 1 km，再用高分辨率地形做 RC/HC。

**检验与出图窗口**为 `24–42°N、105–117.5°E`。这是原 `20–42°N、105–122°E` 里不含海面的取整方框：南界收到 24°N 以避开南部海岸，东界收到 117.5°E 以避开渤海。窗口内地形高度 `> 0` 的格点才进入订正结果、出图和指标。

`model_resolution` 仍填 **20000 m**。中东部调度把 `ABSOLUTE_CORRECTION_TOL_RUN` 设为 **0.08**（模块默认 0.04），用来收窄 10 m 粗糙度订正；`HREF_SCALE_RUN`、`DX_MAX_FACTOR_RUN`、`DX_MIN_FACTOR_RUN` 仍用模块默认。

**时次**：EC 为世界时，文件名 `YYYYMMDDHH.TTT.nc`。实况 HRCLDAS 文件名是北京时，取有效世界时加 8 小时。列表和后面的单元格必须在同一次内核会话里运行。


## 1. 准备

In [ ]:
from __future__ import annotations

import sys
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import meteva_base as meb
import xarray as xr

# 包根 = nbs 的上一级；00temp 用于 import
PKG = Path("..").resolve()
sys.path[:0] = [str(PKG.parent), str(PKG)]

from orographic_wind_downscaling.src.wind_downscaling import (
    RoughnessCorrection,
    ABSOLUTE_CORRECTION_TOL,
    HREF_SCALE,
    DX_MAX_FACTOR,
    DX_MIN_FACTOR,
)

DATA = PKG / "resource" / "data"
OBS_ROOT = DATA / "hrcldas_wind"

WIND_TPL = str(DATA / "wind" / "YYYY" / "YYYYMMDD" / "YYYYMMDDHH.TTT.nc")

SIGMA = str(DATA / "sigma.nc")
TARGET_OROG = str(DATA / "HGT_M_1KM_CHINA.nc")
SILHOUETTE = str(DATA / "silhouette_roughness.nc")
Z0 = str(DATA / "vegetative_roughness_z0.nc")
STANDARD_OROG = str(DATA / "standard_orography_1km.nc")
MODEL_RESOLUTION = 20000.0
OUTPUT_HEIGHT_M = 10.0
LAT_CHUNK = 150

# 中东部：ε 用 0.08，使中等起伏的陆地不再改写 10 m；丘陵仍保留粗糙度订正。其余与模块默认一致。
ABSOLUTE_CORRECTION_TOL_RUN = 0.08  # 模块默认 ABSOLUTE_CORRECTION_TOL = 0.04
HREF_SCALE_RUN = HREF_SCALE  # 默认 2.0
DX_MAX_FACTOR_RUN = DX_MAX_FACTOR  # 默认 3.0
DX_MIN_FACTOR_RUN = DX_MIN_FACTOR  # 默认 0.5

# 无海面方框：南界 24°N，东界 117.5°E。
EVAL_LAT = (24.0, 42.0)
EVAL_LON = (105.0, 117.5)

FIG = DATA / "output" / "figs"
FIG.mkdir(parents=True, exist_ok=True)

cases = []
for path in sorted((DATA / "wind").glob("????/????????/*.nc")):
    init_text, lead_text = path.stem.split(".")
    cases.append((datetime.strptime(init_text, "%Y%m%d%H"), int(lead_text)))
print("n cases", len(cases))
print("example wind:", meb.get_path(WIND_TPL, cases[0][0], cases[0][1]))

## 2. 批量订正

`corrected` 与 `cases` 顺序一致，每个元素是 `24–42°N、105–117.5°E` 的 10 m 订正结果，地形高度不大于 0 的格点为缺测。静态场裁到检验区后只读一次；算法强度由准备格中的 `*_RUN` 参数控制，其中 ε 为 0.08。


In [ ]:
from scipy.ndimage import map_coordinates


def _crop_eval(field: xr.DataArray) -> xr.DataArray:
    return field.sel(lat=slice(*EVAL_LAT), lon=slice(*EVAL_LON))


sigma = _crop_eval(meb.read_griddata_from_nc(SIGMA))
target = _crop_eval(meb.read_griddata_from_nc(TARGET_OROG))
standard = _crop_eval(meb.read_griddata_from_nc(STANDARD_OROG))
silhouette = _crop_eval(meb.read_griddata_from_nc(SILHOUETTE))
z0 = _crop_eval(meb.read_griddata_from_nc(Z0))
ppres = RoughnessCorrection.infer_grid_resolution_from_coords(target)
n_lat = int(target.sizes["lat"])
n_lon = int(target.sizes["lon"])
lat_1km = np.asarray(target.lat.values, dtype=np.float64)
lon_1km = np.asarray(target.lon.values, dtype=np.float64)
land = np.squeeze(np.asarray(target.values, dtype=np.float32)) > 0
if land.ndim != 2 or land.shape != (n_lat, n_lon):
    raise ValueError(f"陆地掩码应为 ({n_lat}, {n_lon})，实际 {land.shape}")
print(
    "eval", EVAL_LAT, EVAL_LON, "ppres_m", round(ppres, 1),
    "n_lat", n_lat, "n_lon", n_lon, "land_frac", round(float(land.mean()), 3),
)


def _slab(field: xr.DataArray, sl: slice) -> np.ndarray:
    """取出纬度块，保留 (lat, lon)。先丢掉非空间的长度为 1 的维。"""
    da = field.isel(lat=sl)
    drop = [d for d in da.dims if d not in ("lat", "lon") and da.sizes[d] == 1]
    if drop:
        da = da.squeeze(drop, drop=True)
    arr = np.asarray(da.values, dtype=np.float32)
    if arr.ndim != 2:
        raise ValueError(f"_slab 后应为 (lat, lon)，实际 dims={da.dims} shape={arr.shape}")
    return arr


def _load_wind(wind_path: str):
    with xr.open_dataset(wind_path) as wind_ds:
        speed = np.squeeze(np.asarray(wind_ds["wind_speed"].values, dtype=np.float32))
        levels = np.asarray(wind_ds["level"].values, dtype=np.float32)
        lat = np.asarray(wind_ds["lat"].values, dtype=np.float64)
        lon = np.asarray(wind_ds["lon"].values, dtype=np.float64)
    return speed, levels, lat, lon


def _sample_coords(lat_src, lon_src, lat_dst, lon_dst) -> np.ndarray:
    y = (lat_dst - lat_src[0]) / (lat_src[1] - lat_src[0])
    x = (lon_dst - lon_src[0]) / (lon_src[1] - lon_src[0])
    y = np.clip(y, 0.0, lat_src.size - 1)
    x = np.clip(x, 0.0, lon_src.size - 1)
    yy, xx = np.meshgrid(y, x, indexing="ij")
    return np.vstack((yy.ravel(), xx.ravel()))


def _bilinear(plane: np.ndarray, coords: np.ndarray, nlat: int, nlon: int) -> np.ndarray:
    return map_coordinates(plane, coords, order=1, mode="nearest").reshape(nlat, nlon).astype(np.float32)


def interp_10m(wind_path: str) -> np.ndarray:
    """把 0.1° 的 10 m 风速双线性插到中东部 1 km 网格。"""
    speed, levels, lat_src, lon_src = _load_wind(wind_path)
    level_index = int(np.where(np.isclose(levels, OUTPUT_HEIGHT_M))[0][0])
    out = np.empty((n_lat, n_lon), dtype=np.float32)
    for i0 in range(0, n_lat, LAT_CHUNK):
        i1 = min(i0 + LAT_CHUNK, n_lat)
        coords = _sample_coords(lat_src, lon_src, lat_1km[i0:i1], lon_1km)
        out[i0:i1] = _bilinear(speed[level_index], coords, i1 - i0, n_lon)
    out = np.where(land, out, np.nan).astype(np.float32)
    return out


def correct_10m(wind_path: str) -> xr.DataArray:
    """中东部：插到 1 km 后按原算法订正，只返回 10 m。"""
    speed, levels, lat_src, lon_src = _load_wind(wind_path)
    level_index = int(np.where(np.isclose(levels, OUTPUT_HEIGHT_M))[0][0])
    out = np.empty((n_lat, n_lon), dtype=np.float32)
    for i0 in range(0, n_lat, LAT_CHUNK):
        i1 = min(i0 + LAT_CHUNK, n_lat)
        sl = slice(i0, i1)
        coords = _sample_coords(lat_src, lon_src, lat_1km[sl], lon_1km)
        wind_values = np.empty((speed.shape[0], i1 - i0, n_lon), dtype=np.float32)
        for k in range(speed.shape[0]):
            wind_values[k] = _bilinear(speed[k], coords, i1 - i0, n_lon)
        plugin = RoughnessCorrection(
            _slab(silhouette, sl),
            _slab(sigma, sl),
            _slab(target, sl),
            _slab(standard, sl),
            MODEL_RESOLUTION,
            ppres=ppres,
            z0=_slab(z0, sl),
            absolute_correction_tol=ABSOLUTE_CORRECTION_TOL_RUN,
            href_scale=HREF_SCALE_RUN,
            dx_max_factor=DX_MAX_FACTOR_RUN,
            dx_min_factor=DX_MIN_FACTOR_RUN,
        )
        result, _hc = plugin.process(wind_values, height_grid=levels)
        out[i0:i1] = result[level_index]
        del result, plugin, wind_values
    out = np.where(land, out, np.nan).astype(np.float32)
    return meb.grid_data(grid1km, data=out)


grid1km = meb.get_grid_of_data(target)
corrected = []
for t0, lead in cases:
    corrected.append(correct_10m(meb.get_path(WIND_TPL, t0, lead)))
    print("corr", t0.strftime("%Y%m%d%H") + f".{lead:03d}", "n", len(corrected))
print("correct done", len(corrected))


## 3. 三张图（抽第 1 个时次的 10 m，中东部）

1 订正前　2 订正后　3 实况。三张图共用同一套色标，范围已裁到检验区。


In [ ]:
from meteva_base.tool.plot_tools import pcolormesh_2d_grid

t0, lead = cases[0]
tag = t0.strftime("%Y%m%d%H") + f".{lead:03d}"
valid = t0 + timedelta(hours=lead)
bjt = valid + timedelta(hours=8)
step = 10

before = meb.grid_data(
    grid1km,
    data=interp_10m(meb.get_path(WIND_TPL, t0, lead)),
)
before = before.isel(lat=slice(None, None, step), lon=slice(None, None, step))
before.name = "wind_speed"
before.attrs["units"] = "m s-1"

after = corrected[0].isel(lat=slice(None, None, step), lon=slice(None, None, step))
after.name = "wind_speed"
after.attrs["units"] = "m s-1"

day, hhmm = bjt.strftime("%Y%m%d"), bjt.strftime("%H%M")
obs_path = next((OBS_ROOT / bjt.strftime("%Y") / day).glob(f"*WIND-{day}{hhmm}.GRB2"))
obs_da = xr.open_dataset(str(obs_path), engine="cfgrib")["si10"]
dest = target
obs_da = obs_da.sel(
    latitude=slice(float(dest.lat.min()), float(dest.lat.max())),
    longitude=slice(float(dest.lon.min()), float(dest.lon.max())),
).reindex(
    latitude=dest.lat.values,
    longitude=dest.lon.values,
    method="nearest",
    tolerance=0.006,
)
obs_values = np.where(land, np.asarray(obs_da.values, dtype=np.float32), np.nan)
obs = meb.grid_data(meb.get_grid_of_data(dest), data=obs_values)
obs = obs.isel(lat=slice(None, None, step), lon=slice(None, None, step))
obs.name = "wind_speed"
obs.attrs["units"] = "m s-1"

both = np.concatenate([
    np.asarray(before.values, dtype=float).ravel(),
    np.asarray(after.values, dtype=float).ravel(),
    np.asarray(obs.values, dtype=float).ravel(),
])
both = both[np.isfinite(both)]
vmin = np.floor(np.nanpercentile(both, 1))
vmax = np.ceil(np.nanpercentile(both, 99))
cstep = 1.0
clevs = np.arange(vmin, vmax + 0.1, cstep)
while len(clevs) >= 20:
    cstep *= 2
    clevs = np.arange(vmin, vmax + 0.1, cstep)
clevs = clevs.tolist()

pcolormesh_2d_grid(
    before, save_path=str(FIG / f"{tag}_1_before.png"),
    title=f"1 订正前 10 m（m/s） {tag}", clevs=clevs, cmap="YlOrRd", show=True,
)
pcolormesh_2d_grid(
    after, save_path=str(FIG / f"{tag}_2_after.png"),
    title=f"2 订正后 10 m（m/s） {tag}", clevs=clevs, cmap="YlOrRd", show=True,
)
pcolormesh_2d_grid(
    obs, save_path=str(FIG / f"{tag}_3_obs.png"),
    title=f"3 实况 HRCLDAS 10 m（m/s） BJT {bjt:%Y%m%d%H%M}",
    clevs=clevs, cmap="YlOrRd", show=True,
)
print("figs:", FIG)

## 4. 验证 Bias / RMSE（订正前 / 订正后 vs 实况）

- **订正前**：0.1° 风速插到 1 km 后的 10 m。
- **订正后**：`corrected` 对应时次的 10 m。
- 实况为 HRCLDAS 10 m，均在中东部检验区内评分。ΔRMSE = 订正后 − 订正前；持平可接受，重点看是否明显偏大。


In [ ]:
def _bias_rmse(pred: np.ndarray, obs: np.ndarray) -> tuple[float, float]:
    mask = np.isfinite(pred) & np.isfinite(obs)
    diff = pred[mask] - obs[mask]
    return float(np.mean(diff)), float(np.sqrt(np.mean(diff**2)))


if len(corrected) != len(cases):
    raise RuntimeError(f"corrected 有 {len(corrected)} 个时次，cases 有 {len(cases)} 个")

dest = target
rows = []
for (t0, lead), after in zip(cases, corrected):
    tag = t0.strftime("%Y%m%d%H") + f".{lead:03d}"
    bjt = t0 + timedelta(hours=lead + 8)
    day, hhmm = bjt.strftime("%Y%m%d"), bjt.strftime("%H%M")
    obs_path = next((OBS_ROOT / bjt.strftime("%Y") / day).glob(f"*WIND-{day}{hhmm}.GRB2"))
    obs_da = xr.open_dataset(str(obs_path), engine="cfgrib")["si10"]
    obs_da = obs_da.sel(
        latitude=slice(float(dest.lat.min()), float(dest.lat.max())),
        longitude=slice(float(dest.lon.min()), float(dest.lon.max())),
    ).reindex(
        latitude=dest.lat.values,
        longitude=dest.lon.values,
        method="nearest",
        tolerance=0.006,
    )

    before_values = interp_10m(meb.get_path(WIND_TPL, t0, lead)).ravel()
    obs_values = np.where(land, np.asarray(obs_da.values, dtype=float), np.nan).ravel()
    after_values = np.asarray(after.values, dtype=float).ravel()
    bias0, rmse0 = _bias_rmse(before_values, obs_values)
    bias1, rmse1 = _bias_rmse(after_values, obs_values)
    rows.append((tag, bias0, rmse0, bias1, rmse1))
    print(
        f"{tag}  before bias={bias0:.3f} m/s rmse={rmse0:.3f} m/s | "
        f"after bias={bias1:.3f} m/s rmse={rmse1:.3f} m/s | "
        f"Δrmse={rmse1 - rmse0:+.3f} m/s"
    )

mean_b0 = float(np.mean([row[1] for row in rows]))
mean_r0 = float(np.mean([row[2] for row in rows]))
mean_b1 = float(np.mean([row[3] for row in rows]))
mean_r1 = float(np.mean([row[4] for row in rows]))
print(
    f"mean  before bias={mean_b0:.3f} m/s rmse={mean_r0:.3f} m/s | "
    f"after bias={mean_b1:.3f} m/s rmse={mean_r1:.3f} m/s | "
    f"Δrmse={mean_r1 - mean_r0:+.3f} m/s"
)

上一格输出中东部检验区的 bias / RMSE。
